# 📘 Módulo 4 — Introdução às Distribuições de Probabilidade
## Livro Didático Aplicado (Híbrido)

- 🔵 **Conteúdo oficial do módulo 4 (IBM)**
- 🟣 **Conteúdo expandido (Livro Didático)**
- 🟠 **Conteúdo avançado (Opcional, matemático)**

Este notebook foi pensado como:
- material de estudo profundo,
- alinhado ao curso IBM,
- mas com muito mais teoria, exemplos, simulações e demonstrações.

Você pode:
- seguir só o que é 🔵 (curso),
- ou mergulhar no 🟣 (livro didático),
- e abrir o 🟠 (avançado) quando quiser ir mais fundo.

# 📚 Índice

1. [Probabilidade e variáveis aleatórias](#1-probabilidade-e-variáveis-aleatórias)  
2. [Distribuições discretas](#2-distribuições-discretas)  
3. [Distribuições contínuas](#3-distribuições-contínuas)  
4. [Distribuição normal e Z-score](#4-distribuição-normal-e-z-score)  
5. [Probabilidades aplicadas com teaching ratings](#5-probabilidades-aplicadas-com-teaching-ratings)  
6. [Teste de hipótese com normal](#6-teste-de-hipótese-com-normal)  
7. [Exercícios guiados](#7-exercícios-guiados)  
8. [Diagramas conceituais](#8-diagramas-conceituais)  
9. [Apêndice matemático avançado](#9-apêndice-matemático-avançado)  

---
# 0. Setup — bibliotecas e dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from math import sqrt

plt.style.use("seaborn-v0_8-whitegrid")

🔵 **Conteúdo oficial do módulo 4 (IBM)**  
Usaremos o mesmo dataset do curso: *teaching ratings*.

In [ ]:
ratings_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/teachingratings.csv"
ratings_df = pd.read_csv(ratings_url)
ratings_df.head()

---
# 1. Probabilidade e variáveis aleatórias
<a id="1-probabilidade-e-variáveis-aleatórias"></a>

🟣 **Conteúdo expandido (Livro Didático)**

Vamos começar pela base conceitual:

- Probabilidade  
- Espaço amostral  
- Eventos  
- Variáveis aleatórias  
- Distribuições de probabilidade  

Isso conecta diretamente com o vídeo do curso que fala de:

> “probabilidade é uma medida entre 0 e 1… espaço de probabilidade… lançar dois dados…”

## 1.1 Probabilidade e espaço amostral

- **Espaço amostral** $ \Omega $: conjunto de todos os resultados possíveis.
- **Evento**: subconjunto de $ \Omega $.
- **Probabilidade**: função $ P: \mathcal{F} \to [0,1] $ que satisfaz:
  1. $ P(\Omega) = 1 $
  2. Se $ A_1, A_2, \dots $ são eventos mutuamente exclusivos,  
     então

$$
P\left(\bigcup_i A_i\right) = \sum_i P(A_i).
$$

Exemplo simples:
- Lançar um dado justo:
  - $ \Omega = \{1,2,3,4,5,6\} $
  - $ P(\{k\}) = 1/6 $ para cada face.

## 1.2 Dois dados — espaço amostral e distribuição da soma

🔵 **Conteúdo oficial do módulo 4 (IBM)**  
O curso usa o exemplo de dois dados para ilustrar:
- espaço de probabilidade,
- distribuição de probabilidade,
- soma dos dois dados,
- probabilidade de obter 2, 3, 4, …, 12.

In [ ]:
# 🟣 Simulação e construção da distribuição da soma de dois dados

outcomes = [(i, j) for i in range(1, 7) for j in range(1, 7)]
sums = [i + j for (i, j) in outcomes]

values, counts = np.unique(sums, return_counts=True)
probs = counts / counts.sum()

for v, c, p in zip(values, counts, probs):
    print(f"Soma = {v:2d} | contagem = {c:2d} | prob = {p:.3f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(values, probs, color="steelblue")
plt.xlabel("Soma dos dois dados")
plt.ylabel("Probabilidade")
plt.title("Distribuição de probabilidade da soma de dois dados")
plt.xticks(values)
plt.show()

🟣 **Interpretação**

- A soma 7 é a mais provável (6 maneiras de obtê-la).
- As probabilidades somam 1.
- Se quisermos $ P(\text{soma} \le 6) $, somamos as barras até 6.

Isso nos leva à **função de distribuição acumulada (CDF)**.

In [ ]:
cdf_vals = probs.cumsum()
for v, cdf in zip(values, cdf_vals):
    print(f"P(Soma ≤ {v}) = {cdf:.3f}")

## 1.3 Variáveis aleatórias

🟣 **Definição intuitiva**

Uma **variável aleatória** é uma função que associa um número a cada resultado de um experimento aleatório.

- Discreta: assume valores contáveis (0,1,2,3,…).
- Contínua: assume valores em intervalos (por exemplo, qualquer real entre 0 e 1).

A distribuição de probabilidade de uma variável aleatória descreve:
- quais valores ela pode assumir,
- com que probabilidade (discreta) ou densidade (contínua).

<details>
<summary>🟠 Conteúdo avançado (opcional) — definição formal de variável aleatória</summary>

Uma variável aleatória é uma função mensurável:

$$
X: (\Omega, \mathcal{F}, P) \to (\mathbb{R}, \mathcal{B})
$$

onde:
- $ (\Omega, \mathcal{F}, P) $ é um espaço de probabilidade,
- $ \mathcal{B} $ é a σ-álgebra de Borel em $ \mathbb{R} $.

A distribuição de $X$ é a medida de probabilidade induzida:

$$
P_X(B) = P(X^{-1}(B)), \quad B \in \mathcal{B}.
$$

Isso conecta probabilidade com teoria da medida.

</details>

---
# 2. Distribuições discretas
<a id="2-distribuições-discretas"></a>

🟣 **Conteúdo expandido (Livro Didático)**  
O módulo 4 foca na normal, mas para ter uma visão sólida de distribuições,
vale muito entender as discretas:

- Bernoulli  
- Binomial  
- Poisson  

Elas aparecem fortemente em módulos seguintes (testes de hipótese, regressão, etc.).

## 2.1 Bernoulli

- Um único experimento com dois resultados: sucesso (1) ou fracasso (0).
- Parâmetro: $ p = P(X=1) $.

PMF:

$$
P(X = 1) = p,\quad P(X = 0) = 1-p
$$

Média:

$$
E[X] = p
$$

Variância:

$$
Var(X) = p(1-p)
$$

In [ ]:
p = 0.3
n_samples = 10000
bernoulli_samples = np.random.binomial(n=1, p=p, size=n_samples)

plt.figure(figsize=(5, 3))
plt.hist(bernoulli_samples, bins=[-0.5, 0.5, 1.5], rwidth=0.8, color="skyblue")
plt.xticks([0, 1])
plt.title(f"Bernoulli(p={p})")
plt.xlabel("Valor")
plt.ylabel("Frequência")
plt.show()

print("Média empírica:", bernoulli_samples.mean())
print("Média teórica:", p)
print("Variância empírica:", bernoulli_samples.var())
print("Variância teórica:", p * (1 - p))

## 2.2 Binomial

- Número de sucessos em $ n $ tentativas independentes, cada uma com probabilidade $ p $ de sucesso.

PMF:

$$
P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}
$$

Média:

$$
E[X] = np
$$

Variância:

$$
Var(X) = np(1-p)
$$

In [ ]:
n, p = 20, 0.4
n_samples = 10000
binomial_samples = np.random.binomial(n=n, p=p, size=n_samples)

plt.figure(figsize=(7, 3))
plt.hist(binomial_samples, bins=range(0, n+2), align="left", rwidth=0.8, color="lightgreen")
plt.title(f"Binomial(n={n}, p={p})")
plt.xlabel("Número de sucessos")
plt.ylabel("Frequência")
plt.show()

print("Média empírica:", binomial_samples.mean())
print("Média teórica:", n * p)
print("Variância empírica:", binomial_samples.var())
print("Variância teórica:", n * p * (1 - p))

## 2.3 Poisson

- Número de eventos em um intervalo de tempo ou espaço, com taxa média $ \lambda $.

PMF:

$$
P(X = k) = \frac{e^{-\lambda} \lambda^k}{k!}
$$

Média e variância:

$$
E[X] = Var(X) = \lambda
$$

In [ ]:
lam = 4
n_samples = 10000
poisson_samples = np.random.poisson(lam=lam, size=n_samples)

plt.figure(figsize=(7, 3))
plt.hist(poisson_samples, bins=range(0, lam*3+2), align="left", rwidth=0.8, color="orange")
plt.title(f"Poisson(λ={lam})")
plt.xlabel("Número de eventos")
plt.ylabel("Frequência")
plt.show()

print("Média empírica:", poisson_samples.mean())
print("Média teórica:", lam)
print("Variância empírica:", poisson_samples.var())
print("Variância teórica:", lam)

---
# 3. Distribuições contínuas
<a id="3-distribuições-contínuas"></a>

🟣 **Conteúdo expandido (Livro Didático)**  
Vamos ver:

- Uniforme  
- Exponencial  
- Normal (que é o foco do módulo 4)

## 3.1 Uniforme contínua

- Intervalo $[a, b]$  
- Todos os valores são igualmente prováveis.

PDF:

$$
f(x) = \frac{1}{b-a}, \quad a \le x \le b
$$

In [ ]:
a, b = 0, 10
n_samples = 10000
uniform_samples = np.random.uniform(low=a, high=b, size=n_samples)

plt.figure(figsize=(7, 3))
plt.hist(uniform_samples, bins=20, density=True, alpha=0.6, color="purple", edgecolor="black")
x_vals = np.linspace(a, b, 200)
plt.plot(x_vals, [1/(b-a)]*len(x_vals), color="red", linewidth=2)
plt.title(f"Uniforme({a}, {b})")
plt.xlabel("x")
plt.ylabel("densidade")
plt.show()

print("Média empírica:", uniform_samples.mean())
print("Média teórica:", (a + b) / 2)

## 3.2 Exponencial

- Tempo até um evento, com taxa $ \lambda $.

PDF:

$$
f(x) = \lambda e^{-\lambda x}, \quad x \ge 0
$$

Média:

$$
E[X] = \frac{1}{\lambda}
$$

In [ ]:
lam = 0.5
n_samples = 10000
exp_samples = np.random.exponential(scale=1/lam, size=n_samples)

plt.figure(figsize=(7, 3))
plt.hist(exp_samples, bins=30, density=True, alpha=0.6, color="darkred", edgecolor="black")
x_vals = np.linspace(0, exp_samples.max(), 200)
plt.plot(x_vals, lam * np.exp(-lam * x_vals), color="black", linewidth=2)
plt.title(f"Exponencial(λ={lam})")
plt.xlabel("x")
plt.ylabel("densidade")
plt.show()

print("Média empírica:", exp_samples.mean())
print("Média teórica:", 1 / lam)

---
# 4. Distribuição normal e Z-score
<a id="4-distribuição-normal-e-z-score"></a>

🔵 **Conteúdo oficial do módulo 4 (IBM)**  
O módulo 4 enfatiza:

- distribuição normal,  
- normal padrão,  
- fórmula da PDF,  
- Z-score,  
- uso de `scipy.stats.norm`,  
- visualização da curva normal.

## 4.1 Definição da normal

Uma variável $X$ tem distribuição normal com média $\mu$ e desvio padrão $\sigma$ se sua PDF é:

$$
f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}
$$

Notação: $ X \sim N(\mu, \sigma^2) $.

In [ ]:
from scipy.stats import norm

x_axis = np.arange(-4, 4, 0.1)
plt.figure(figsize=(8, 4))
plt.plot(x_axis, norm.pdf(x_axis, loc=0, scale=1))
plt.title("Normal padrão N(0, 1)")
plt.xlabel("x")
plt.ylabel("densidade")
plt.show()

## 4.2 Normal padrão e Z-score

🔵 **Conteúdo oficial do módulo 4 (IBM)**  
O curso mostra:
- normal padrão (média 0, desvio 1),
- padronização via Z-score.

Z-score:

$$
Z = \frac{X - \mu}{\sigma}
$$

Isso transforma qualquer normal $ N(\mu, \sigma^2) $ em $ N(0, 1) $.

<details>
<summary>🟠 Conteúdo avançado (opcional) — transformação linear de normais</summary>

Se $ X \sim N(\mu, \sigma^2) $ e definimos:

$$
Z = \frac{X - \mu}{\sigma},
$$

então:

- $E[Z] = 0$,  
- $Var(Z) = 1$,  
- e a PDF de $Z$ é a normal padrão.

Isso decorre da transformação linear e da regra de mudança de variável em integrais.

</details>

---
# 5. Probabilidades aplicadas com teaching ratings
<a id="5-probabilidades-aplicadas-com-teaching-ratings"></a>

🔵 **Conteúdo oficial do módulo 4 (IBM)**  
O lab do módulo 4 faz:

- cálculo da média e desvio padrão de `eval`,  
- cálculo de probabilidades usando `norm.cdf`,  
- exemplos:
  - $P(\text{eval} > 4.5)$  
  - $P(3.5 < \text{eval} < 4.2)$  

In [ ]:
eval_mean = round(ratings_df["eval"].mean(), 3)
eval_sd = round(ratings_df["eval"].std(), 3)
eval_mean, eval_sd

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(ratings_df["eval"], bins=20, density=True, alpha=0.6, color="skyblue", edgecolor="black")

x_vals = np.linspace(ratings_df["eval"].min(), ratings_df["eval"].max(), 200)
plt.plot(x_vals, norm.pdf(x_vals, loc=eval_mean, scale=eval_sd), color="red", linewidth=2)

plt.title(f"Distribuição dos evaluation scores (média={eval_mean}, sd={eval_sd})")
plt.xlabel("evaluation score")
plt.ylabel("densidade")
plt.show()

## 5.1 Probabilidade de evaluation score > 4.5

🔵 **Pergunta do lab:**
> Usando o teachers' rating dataset, qual é a probabilidade de receber um evaluation score maior que 4.5?

In [ ]:
z_45 = (4.5 - eval_mean) / eval_sd
prob_greater_45 = 1 - norm.cdf(z_45)
print("Z-score para 4.5:", round(z_45, 3))
print("P(X > 4.5):", round(prob_greater_45, 4))

## 5.2 Probabilidade de 3.5 < evaluation score < 4.2

🔵 **Pergunta do lab:**
> Qual é a probabilidade de receber um evaluation score maior que 3.5 e menor que 4.2?

In [ ]:
x1, x2 = 3.5, 4.2
prob_less_x1 = norm.cdf((x1 - eval_mean) / eval_sd)
prob_less_x2 = norm.cdf((x2 - eval_mean) / eval_sd)
prob_between = prob_less_x2 - prob_less_x1

print("P(X < 3.5):", round(prob_less_x1, 4))
print("P(X < 4.2):", round(prob_less_x2, 4))
print("P(3.5 < X < 4.2):", round(prob_between, 4), "→", round(prob_between * 100, 1), "%")

---
# 6. Teste de hipótese com normal
<a id="6-teste-de-hipótese-com-normal"></a>

🔵 **Conteúdo oficial do módulo 4 (IBM)**  
O módulo traz:

- definição de hipótese nula e alternativa,  
- alfa (nível de significância),  
- p-value,  
- exemplo com jogadores de basquete,  
- uso da normal para média amostral quando $\sigma$ é conhecido.

## 6.1 Exemplo dos jogadores de basquete

Cenário:
- Média histórica: 12 pontos por jogo, $\sigma = 5.5$.  
- Amostra de 36 jogadores regionais: média $= 10.7$.  
- Teste:
  - $H_0: \mu = 12$  
  - $H_1: \mu \neq 12$ (two-tailed)

Estatística de teste:

$$
Z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}}
$$

In [ ]:
mu0 = 12
sigma = 5.5
xbar = 10.7
n = 36

z_stat = (xbar - mu0) / (sigma / sqrt(n))
p_value_two_tailed = 2 * norm.cdf(z_stat)  # z_stat < 0, usamos cauda inferior e multiplicamos por 2

print("Z-statistic:", round(z_stat, 3))
print("p-value (two-tailed):", round(p_value_two_tailed, 3))

Se adotarmos $ \alpha = 0.05 $:
- Se p-value < 0.05 → rejeitamos $H_0$.  
- Se p-value ≥ 0.05 → não rejeitamos $H_0$.

Aqui, p-value > 0.05 → **não rejeitamos $H_0$**.  
Não há evidência suficiente para dizer que a média dos jogadores regionais é diferente da média histórica.

---
# 7. Exercícios guiados
<a id="7-exercícios-guiados"></a>

🟣 **Exercícios para fixação e raciocínio**

Use estes exercícios como treino para:
- provas do curso,  
- exame final,  
- e para consolidar o raciocínio estatístico.

### Exercício 1 — Binomial

Considere $ X \sim Binomial(n=20, p=0.4) $.

1. Simule 10.000 valores de $X$.  
2. Plote o histograma.  
3. Compare a média e a variância empíricas com as teóricas.  
4. Interprete o formato da distribuição.

In [ ]:
# TODO: seu código aqui (pode se basear na seção 2.2)

### Exercício 2 — Exponencial

Considere $ X \sim Exponencial(\lambda = 0.5) $.

1. Simule 10.000 valores.  
2. Plote o histograma com densidade.  
3. Compare a média empírica com $1/\lambda$.  
4. A distribuição é simétrica ou assimétrica?

In [ ]:
# TODO: seu código aqui (pode se basear na seção 3.2)

### Exercício 3 — Comparando duas normais

Considere:
- $ X \sim N(0, 1) $  
- $ Y \sim N(5, 2^2) $

1. Simule 10.000 valores de cada.  
2. Plote as duas distribuições no mesmo gráfico (KDE ou histograma).  
3. Compare:
   - centros (médias),  
   - dispersões (desvios padrão).  
4. Relacione com interpretação de média e desvio padrão em problemas reais.

In [ ]:
# TODO: seu código aqui

### Exercício 4 — Probabilidade com teaching ratings

Usando o dataset `ratings_df`:

1. Calcule $P(\text{eval} > 3.3)$.  
2. Calcule $P(2 < \text{eval} < 3)$.  
3. Interprete os resultados em termos de “porcentagem de professores”.

In [ ]:
# TODO: seu código aqui
# Dica: use eval_mean, eval_sd e norm.cdf

### Exercício 5 — IQ e teste de hipótese

Para testar a hipótese de que dormir pelo menos 8 horas deixa uma pessoa mais inteligente,
12 pessoas que dormiram 8h/dia por 1 ano tiveram seus IQ testados:

`116, 111, 101, 120, 99, 94, 106, 115, 107, 101, 110, 92`

Teste:
- $H_0: \mu = 100$  
- $H_a: \mu > 100$

1. Calcule a média amostral.  
2. Calcule o desvio padrão amostral.  
3. Calcule a estatística de teste (aprox. normal).  
4. Calcule o p-value.  
5. Conclua para $ \alpha = 0.05 $.

In [ ]:
# TODO: seu código aqui

---
# 8. Diagramas conceituais
<a id="8-diagramas-conceituais"></a>

🟣 **Mapa mental — qual distribuição usar?**

- **Variável discreta (contagem)**:
  - Número de sucessos em $n$ tentativas → Binomial  
  - Número de eventos em intervalo → Poisson  

- **Variável contínua**:
  - Tempo até evento → Exponencial  
  - Valores em intervalo $[a, b]$ igualmente prováveis → Uniforme  
  - Valores em torno de uma média, com simetria → Normal  

Esse tipo de mapa ajuda muito em provas:  
o enunciado descreve o fenômeno, você escolhe a distribuição.

---
# 9. Apêndice matemático avançado
<a id="9-apêndice-matemático-avançado"></a>

<details>
<summary>🟠 Derivação da forma da PDF da normal (esboço)</summary>

A forma:

$$
f(x) = C e^{-ax^2}
$$

surge naturalmente ao exigir:
- simetria em torno de 0,  
- decaimento suave,  
- integrabilidade.  

A constante de normalização $C$ é escolhida para que:

$$
\int_{-\infty}^{\infty} f(x)\,dx = 1.
$$

O cálculo clássico mostra que:

$$
\int_{-\infty}^{\infty} e^{-x^2/2}\,dx = \sqrt{2\pi},
$$

o que leva à forma:

$$
f(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}
$$

para a normal padrão.

</details>

<details>
<summary>🟠 Teorema Central do Limite (visão intuitiva)</summary>

O Teorema Central do Limite (TCL) diz, grosso modo, que:

- Se $X_1, X_2, \dots$ são i.i.d. com média $\mu$ e variância $\sigma^2$,  
- então a média amostral:

$$
\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i
$$

tende, em distribuição, a uma normal:

$$
\sqrt{n}\left(\bar{X}_n - \mu\right) \xrightarrow{d} N(0, \sigma^2).
$$

Isso explica por que a normal aparece tanto:
- somas de muitas variáveis independentes “tendem” à normal.

</details>

---
## Encerramento

Você tem agora:

- a base conceitual de probabilidade e variáveis aleatórias,  
- uma visão ampla de distribuições discretas e contínuas,  
- o foco do módulo 4 (normal, Z-score, CDF, probabilidades),  
- exemplos aplicados com o dataset do curso,  
- um exemplo de teste de hipótese com normal,  
- exercícios para fixação,  
- e um apêndice matemático para quando quiser ir mais fundo.

Este arquivo pode ser:
- seu notebook autoral do módulo 4,  
- base para o capítulo 4 do Companion Book,  
- material de revisão para o exame final.